In [2]:
%pip install pymupdf tqdm

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.8 MB 1.5 MB/s eta 0:00:13
   - -------------------------------------- 0.8/19.8 MB 1.5 MB/s eta 0:00:13
   -- ------------------------------------- 1.0/19.8 MB 1.2 MB/s eta 0:00:16
   -- ------------------------------------- 1.3/19.8 MB 1.2 MB/s eta 0:00:16
   -- ------------------------------------- 1.3/19.8 MB 1.2 MB/s eta 0:00:16
   --- ------------------------------------ 1.6/19.8 MB 951.9 kB/s eta 0:00:20
   --- ------------------------------------ 1.6/19.8 MB 951.9 kB/s eta 0:00:20
   --- ------------------------------------ 1.8/19.8 MB 977.1 kB/s eta 0:00:19
   ---- ----------------------------------- 2.1/19.8 MB 1.0 MB/s eta 0:00:18
   ---- -------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import os
import fitz  # PyMuPDF
import csv
import hashlib
from pathlib import Path
from tqdm import tqdm

# ==========================================
# KONFIGURASI PENGGUNA - UBAH BAGIAN INI
# ==========================================
# Masukkan lokasi folder fisik PDF di bawah ini (gunakan huruf 'r' di depan tanda kutip)
PDF_SOURCE_DIR = r"C:\Users\Arya\Downloads\Contoh pdf" 

# Output laporan akan otomatis masuk ke folder reports yang tadi kita buat
REPORT_CSV = r"../PDF Extraction/reports/pdf_audit_report.csv"
ERROR_LOG = r"../PDF Extraction/reports/audit_errors.log"

SAMPLE_PAGES = 3
MIN_CHARS_PER_PAGE_DIGITAL = 50 

def get_file_checksum(filepath, chunk_size=8192):
    hasher = hashlib.md5()
    try:
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(chunk_size), b''):
                hasher.update(chunk)
        return hasher.hexdigest()
    except Exception:
        return None

def audit_pdf(filepath):
    audit_data = {
        "file_name": os.path.basename(filepath),
        "file_path": filepath,
        "file_size_kb": round(os.path.getsize(filepath) / 1024, 2),
        "page_count": 0,
        "is_readable": False,
        "needs_ocr": False,
        "checksum": None,
        "status": "Success",
        "error_message": ""
    }
    
    try:
        doc = fitz.open(filepath)
        audit_data["page_count"] = len(doc)
        audit_data["is_readable"] = True
        
        if audit_data["page_count"] > 0:
            pages_to_check = [0, audit_data["page_count"] // 2, audit_data["page_count"] - 1]
            pages_to_check = list(set(pages_to_check)) 
            
            total_text_length = 0
            for page_num in pages_to_check:
                page = doc.load_page(page_num)
                text = page.get_text("text").strip()
                total_text_length += len(text)
                
            avg_chars = total_text_length / len(pages_to_check)
            if avg_chars < MIN_CHARS_PER_PAGE_DIGITAL:
                audit_data["needs_ocr"] = True
                
        doc.close()
        # Hitung checksum hanya jika file berhasil dibaca untuk efisiensi
        audit_data["checksum"] = get_file_checksum(filepath)
        
    except Exception as e:
        audit_data["status"] = "Error"
        audit_data["error_message"] = str(e)
        
    return audit_data

# Eksekusi Utama
print(f"Mencari file PDF di: {PDF_SOURCE_DIR}")
os.makedirs(os.path.dirname(REPORT_CSV), exist_ok=True)

pdf_files = list(Path(PDF_SOURCE_DIR).rglob("*.pdf"))
print(f"Ditemukan {len(pdf_files)} file PDF. Memulai audit...\n")

processed_files = set()
if os.path.exists(REPORT_CSV):
    with open(REPORT_CSV, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            processed_files.add(row["file_path"])
    print(f"Resume: {len(processed_files)} file sudah diaudit sebelumnya.\n")
else:
    with open(REPORT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            "file_name", "file_path", "file_size_kb", "page_count", 
            "is_readable", "needs_ocr", "checksum", "status", "error_message"
        ])
        
with open(REPORT_CSV, 'a', newline='', encoding='utf-8') as f:
    fieldnames = [
        "file_name", "file_path", "file_size_kb", "page_count", 
        "is_readable", "needs_ocr", "checksum", "status", "error_message"
    ]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    for pdf_path in tqdm(pdf_files, desc="Progres Audit"):
        str_path = str(pdf_path)
        if str_path in processed_files:
            continue
            
        result = audit_pdf(str_path)
        writer.writerow(result)
        f.flush()
        
        if result["status"] == "Error":
            with open(ERROR_LOG, 'a', encoding='utf-8') as log_f:
                log_f.write(f"[{result['file_path']}] - {result['error_message']}\n")

print(f"\nAudit selesai! Silakan periksa file {REPORT_CSV}")

Mencari file PDF di: C:\Users\Arya\Downloads\Contoh pdf
Ditemukan 12 file PDF. Memulai audit...



Progres Audit: 100%|██████████| 12/12 [00:00<00:00, 44.24it/s]


Audit selesai! Silakan periksa file ../PDF Extraction/reports/pdf_audit_report.csv


In [2]:
import sys
!{sys.executable} -m pip install pandas

In [3]:
import pandas as pd

# Baca laporan audit
df = pd.read_csv(r"../PDF Extraction/reports/pdf_audit_report.csv")

# Tampilkan ringkasan audit
print("=== RINGKASAN AUDIT PDF ===")
print(f"Total file diproses : {len(df)}")
print(f"Total halaman       : {df['page_count'].sum()}")
print(f"File butuh OCR      : {df['needs_ocr'].sum()}")
print(f"File gagal dibaca   : {(df['status'] == 'Error').sum()}")
print("\nDetail per file:")
df[["file_name", "page_count", "needs_ocr", "status"]]

=== RINGKASAN AUDIT PDF ===
Total file diproses : 12
Total halaman       : 132
File butuh OCR      : 0
File gagal dibaca   : 0

Detail per file:


,file_name,page_count,needs_ocr,status
0,1383-2875-1-SM.pdf,10,False,Success
1,383-Article Text-1294-1-10-20200208.pdf,13,False,Success
2,642-818-1-PB.pdf,9,False,Success
3,872-1588-1-SM.pdf,13,False,Success
4,a549bc2923747cdb92e0ba7dd09a52e2b7ad.pdf,6,False,Success
5,"admin,+Artika+Dewi+21-27.pdf",7,False,Success
6,"admin,+combinepdf+(5).pdf",20,False,Success
7,DISFUNGSI_ENDOTEL_PADA_PREEKLAMPSIA.pdf,7,False,Success
8,Hubungan_Riwayat_Hipertensi_Dan_Status_G.pdf,7,False,Success
9,Literature_Review_Faktor_Faktor_Risiko_T.pdf,11,False,Success


In [4]:
import os
import json
import pymupdf  # Menggunakan sintaks modern pymupdf
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ==========================================
# KONFIGURASI PATH
# ==========================================
AUDIT_REPORT_CSV = r"../PDF Extraction/reports/pdf_audit_report.csv"
OUTPUT_DIR = r"../PDF Extraction/data/extracted"
EXTRACTION_LOG_CSV = r"../PDF Extraction/reports/extraction_summary.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(EXTRACTION_LOG_CSV), exist_ok=True)

# 1. Baca daftar file yang valid dari audit report
df_audit = pd.read_csv(AUDIT_REPORT_CSV)
valid_files = df_audit[df_audit["status"] == "Success"].to_dict("records")

# 2. Cek checkpoint: file mana saja yang sudah diekstrak sebelumnya
extracted_files = set()
if os.path.exists(EXTRACTION_LOG_CSV):
    df_log = pd.read_csv(EXTRACTION_LOG_CSV)
    extracted_files = set(df_log["file_path"].tolist())

extraction_records = []

print(f"Memulai ekstraksi {len(valid_files)} file...")

for item in tqdm(valid_files, desc="Ekstraksi Halaman PDF"):
    pdf_path = item["file_path"]
    file_name = item["file_name"]
    checksum = item["checksum"]

    if pdf_path in extracted_files:
        continue

    extracted_pages = []
    total_chars = 0
    status = "Success"
    error_msg = ""

    try:
        doc = pymupdf.open(pdf_path)
        for page_idx in range(len(doc)):
            page = doc.load_page(page_idx)
            page_text = page.get_text("text").strip()
            total_chars += len(page_text)

            extracted_pages.append({
                "page_number": page_idx + 1,  # 1-indexed untuk konsistensi UI
                "char_count": len(page_text),
                "text": page_text
            })
        doc.close()

        # Simpan hasil ekstraksi per file ke dalam JSON terpisah
        output_file_name = f"{Path(file_name).stem}_{checksum[:8]}.json" if checksum else f"{Path(file_name).stem}.json"
        output_path = os.path.join(OUTPUT_DIR, output_file_name)

        doc_payload = {
            "file_name": file_name,
            "file_path": pdf_path,
            "checksum": checksum,
            "total_pages": len(extracted_pages),
            "total_chars": total_chars,
            "pages": extracted_pages
        }

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(doc_payload, f, ensure_ascii=False, indent=2)

    except Exception as e:
        status = "Error"
        error_msg = str(e)

    # Catat log ekstraksi
    record = {
        "file_name": file_name,
        "file_path": pdf_path,
        "total_pages": len(extracted_pages),
        "total_chars": total_chars,
        "status": status,
        "error_message": error_msg
    }
    extraction_records.append(record)

# Update log CSV
df_new_records = pd.DataFrame(extraction_records)
if not df_new_records.empty:
    if os.path.exists(EXTRACTION_LOG_CSV):
        df_new_records.to_csv(EXTRACTION_LOG_CSV, mode="a", header=False, index=False, encoding="utf-8")
    else:
        df_new_records.to_csv(EXTRACTION_LOG_CSV, index=False, encoding="utf-8")

print(f"\nEkstraksi selesai! Data tersimpan di: {OUTPUT_DIR}")

Memulai ekstraksi 12 file...


Ekstraksi Halaman PDF: 100%|██████████| 12/12 [00:00<00:00, 37.07it/s]


Ekstraksi selesai! Data tersimpan di: ../PDF Extraction/data/extracted


In [5]:
import os
import json
import sqlite3
from pathlib import Path
from tqdm import tqdm

# ==========================================
# KONFIGURASI CHUNKING & DATABASE
# ==========================================
EXTRACTED_DIR = r"../PDF Extraction/data/extracted"
DB_PATH = r"D:\Project\PDF Extraction\data\rag_metadata.db"

# Strategi chunking berbasis karakter/kata
CHUNK_SIZE = 800       # Target panjang karakter per chunk (~150-200 kata)
CHUNK_OVERLAP = 150    # Overlap antar-chunk agar konteks kalimat tidak putus
MIN_CHUNK_CHARS = 50   # Abaikan cuplikan terlalu pendek (header/footer kosong)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# 1. Inisialisasi Database SQLite & FTS5 Virtual Table
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Tabel metadata dokumen
cursor.execute("""
CREATE TABLE IF NOT EXISTS documents (
    doc_id INTEGER PRIMARY KEY AUTOINCREMENT,
    file_name TEXT,
    file_path TEXT UNIQUE,
    checksum TEXT,
    total_pages INTEGER,
    total_chars INTEGER
)
""")

# Tabel relasi chunk (metadata persisten)
cursor.execute("""
CREATE TABLE IF NOT EXISTS chunks (
    chunk_id INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_id INTEGER,
    file_name TEXT,
    page_number INTEGER,
    chunk_index INTEGER,
    char_count INTEGER,
    content TEXT,
    FOREIGN KEY(doc_id) REFERENCES documents(doc_id)
)
""")

# Virtual Table SQLite FTS5 untuk pencarian full-text berkecepatan tinggi
cursor.execute("""
CREATE VIRTUAL TABLE IF NOT EXISTS fts_chunks USING fts5(
    chunk_id UNINDEXED,
    file_name,
    page_number UNINDEXED,
    content
)
""")
conn.commit()

# 2. Fungsi Text Splitter Sederhana & Aman
def split_text_with_overlap(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        
        # Potong mundur ke spasi terdekat agar tidak memotong kata di tengah
        if end < text_len:
            last_space = chunk.rfind(" ")
            if last_space > chunk_size * 0.7:  # Jaga panjang potongan tetap seimbang
                end = start + last_space
                chunk = text[start:end]
                
        chunk_clean = chunk.strip()
        if len(chunk_clean) >= MIN_CHUNK_CHARS:
            chunks.append(chunk_clean)
            
        start = end - overlap
        if start >= text_len - overlap:
            break
            
    return chunks

# 3. Proses Chunking & Indexing
json_files = list(Path(EXTRACTED_DIR).glob("*.json"))
print(f"Memproses {len(json_files)} file JSON hasil ekstraksi...")

# Ambil daftar file yang sudah terindeks (Checkpoint)
cursor.execute("SELECT file_path FROM documents")
indexed_files = {row[0] for row in cursor.fetchall()}

total_chunks_created = 0

for json_path in tqdm(json_files, desc="Chunking & Indexing FTS5"):
    with open(json_path, "r", encoding="utf-8") as f:
        doc_data = json.load(f)
        
    file_path = doc_data["file_path"]
    if file_path in indexed_files:
        continue
        
    file_name = doc_data["file_name"]
    checksum = doc_data.get("checksum", "")
    total_pages = doc_data.get("total_pages", 0)
    total_chars = doc_data.get("total_chars", 0)
    
    # Simpan entri dokumen
    cursor.execute("""
    INSERT INTO documents (file_name, file_path, checksum, total_pages, total_chars)
    VALUES (?, ?, ?, ?, ?)
    """, (file_name, file_path, checksum, total_pages, total_chars))
    doc_id = cursor.lastrowid
    
    # Proses tiap halaman menjadi chunks
    for page in doc_data.get("pages", []):
        page_num = page["page_number"]
        page_text = page["text"]
        
        if not page_text.strip():
            continue
            
        page_chunks = split_text_with_overlap(page_text)
        
        for c_idx, chunk_text in enumerate(page_chunks):
            # Simpan ke tabel relasional
            cursor.execute("""
            INSERT INTO chunks (doc_id, file_name, page_number, chunk_index, char_count, content)
            VALUES (?, ?, ?, ?, ?, ?)
            """, (doc_id, file_name, page_num, c_idx, len(chunk_text), chunk_text))
            chunk_id = cursor.lastrowid
            
            # Simpan ke FTS5 index
            cursor.execute("""
            INSERT INTO fts_chunks (chunk_id, file_name, page_number, content)
            VALUES (?, ?, ?, ?)
            """, (chunk_id, file_name, page_num, chunk_text))
            
            total_chunks_created += 1
            
    conn.commit()

# Verifikasi total data terindeks
cursor.execute("SELECT count(*) FROM chunks")
total_in_db = cursor.fetchone()[0]

print(f"\nIndexing selesai!")
print(f"Total chunk baru ditambahkan: {total_chunks_created}")
print(f"Total chunk aktif di database: {total_in_db}")
print(f"Database tersimpan di: {DB_PATH}")

conn.close()

Memproses 12 file JSON hasil ekstraksi...


Chunking & Indexing FTS5: 100%|██████████| 12/12 [00:00<00:00, 113.39it/s]


Indexing selesai!
Total chunk baru ditambahkan: 617
Total chunk aktif di database: 617
Database tersimpan di: ../PDF Extraction/data/rag_metadata.db


In [4]:
import sys
!{sys.executable} -m pip install google-genai python-dotenv

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   -------------------------- ------------- 0.8/1.2 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 2.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 5.4 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.2 MB 3.0 MB/s eta 0:00:01
   -------------------------------------- - 2.1/2.2 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 3.0 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------- ----------------------------- 1.0/3.8 MB 5.5 MB/s eta 0:00:01
   ------------------- -------------------- 1.8/3.8 MB 5.7 MB/s eta 0:00:01
   --------------------------- ------------ 2.6/3.8 MB 4.4 MB/s eta 0:00:01
   -------------------------------------- - 3.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [9]:
import os
import sqlite3
import time
from dotenv import load_dotenv
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY belum dikonfigurasi di file .env!")

client = genai.Client(api_key=api_key)
DB_PATH = r"D:\Project\PDF Extraction\data\rag_metadata.db"

INDONESIAN_STOPWORDS = {
    "apa", "apakah", "siapa", "mengapa", "kenapa", "bagaimana", "dimana",
    "kapan", "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk",
    "pada", "adalah", "sebagai", "dengan", "atau", "oleh", "dalam", "bisa",
    "dapat", "jelaskan", "sebutkan", "tentang", "tersebut", "ada"
}

def clean_and_build_fts_query(user_query: str) -> str:
    raw_tokens = "".join([c if c.isalnum() or c.isspace() else " " for c in user_query.lower()]).split()
    meaningful = [t for t in raw_tokens if t not in INDONESIAN_STOPWORDS and len(t) > 2]
    if not meaningful:
        meaningful = [t for t in raw_tokens if len(t) > 2]
    return " OR ".join(meaningful) if meaningful else ""

def retrieve_relevant_chunks(query: str, top_k: int = 8):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    fts_query = clean_and_build_fts_query(query)
    if not fts_query:
        conn.close()
        return []
        
    sql = """
    SELECT c.file_name, c.page_number, c.content, bm25(fts_chunks) AS rank
    FROM fts_chunks f
    JOIN chunks c ON f.chunk_id = c.chunk_id
    WHERE fts_chunks MATCH ?
    ORDER BY rank
    LIMIT ?
    """
    
    try:
        cursor.execute(sql, (fts_query, top_k))
        rows = cursor.fetchall()
    except Exception as e:
        print(f"[DEBUG SQL ERROR]: {e}")
        rows = []
    finally:
        conn.close()
        
    results = []
    for row in rows:
        results.append({
            "file_name": row[0],
            "page_number": row[1],
            "text": row[2].strip(),
            "score": round(row[3], 4)
        })
    return results

def ask_rag_verified(query: str, top_k: int = 8, max_retries: int = 3):
    chunks = retrieve_relevant_chunks(query, top_k=top_k)
    
    if not chunks:
        return (
            "Tidak ditemukan potongan teks yang relevan di database lokal untuk kata kunci tersebut.",
            []
        )
        
    context_blocks = []
    for idx, c in enumerate(chunks, 1):
        context_blocks.append(
            f"[Sumber {idx}] Dokumen: {c['file_name']} (Hal. {c['page_number']})\n{c['text']}"
        )
    context_str = "\n\n".join(context_blocks)
    
    prompt = f"""Kamu adalah analis dokumen akademik dan medis yang presisi.
Jawab pertanyaan berikut HANYA berdasarkan KONTEKS DOKUMEN yang tersedia di bawah ini.

Pedoman Jawaban:
1. Utamakan ketepatan faktual. Jangan berasumsi atau mengarang data di luar teks.
2. Jelaskan fakta penting secara runut, padat, dan sertakan rujukan nama file serta nomor halaman.
3. Jika dokumen hanya memuat sebagian informasi, jelaskan fakta yang ada terlebih dahulu dan sebutkan keterbatasannya secara jujur.

KONTEKS DOKUMEN:
{context_str}

PERTANYAAN:
{query}

JAWABAN ANALITIS:"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            return response.text.strip(), chunks
        except Exception:
            time.sleep(2)
            
    return "Layanan AI sedang sibuk. Silakan coba kembali.", chunks

# ==========================================
# INPUT PERTANYAAN USER (SATU KALI JALAN)
# ==========================================
user_query = input("Masukkan pertanyaan Anda: ").strip()

if not user_query:
    print("Pertanyaan tidak boleh kosong.")
else:
    print(f"\nMencari dokumen terkait: '{user_query}'...")
    jawaban, sources = ask_rag_verified(user_query, top_k=8)
    
    print("\n" + "=" * 60)
    print("JAWABAN AI:")
    print("=" * 60)
    print(jawaban)
    
    if sources:
        print("\n" + "-" * 60)
        print("REFERENSI DOKUMEN YANG DIGUNAKAN:")
        print("-" * 60)
        for idx, s in enumerate(sources[:5], 1):
            print(f"[{idx}] {s['file_name']} (Halaman {s['page_number']})")


Mencari dokumen terkait: 'preeklampsia tuh bagaimana si penanganannya?'...

JAWABAN AI:
Berdasarkan konteks dokumen yang diberikan, **informasi mengenai tata cara atau prosedur penanganan medis preeklampsia secara rinci tidak tersedia dalam teks.**

Berikut adalah fakta terdekat dan keterbatasan informasi yang ditemukan dalam dokumen:

1. **Fakta yang Ada dalam Dokumen:**
   * Pada dokumen `Literature_Review_Faktor_Faktor_Risiko_T.pdf` (Hal. 10), penanganan preeklampsia hanya disinggung dalam bentuk judul daftar pustaka/referensi, yaitu: *"Susiana, S. (2019). Angka Kematian Ibu : Faktor Penyebab Dan Upaya Penanganannya"*. Namun, isi atau rincian mengenai bentuk penanganannya tidak dijelaskan dalam teks konteks.
   * Dokumen-dokumen yang tersedia lebih banyak berfokus pada **faktor risiko** dan **statistik kejadian** preeklampsia (seperti riwayat hipertensi, riwayat preeklampsia sebelumnya, konsumsi kalsium, Indeks Massa Tubuh/IMT/obesitas, paritas, dan usia ibu hamil).

2. **Keterbata

In [2]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("=== DAFTAR MODEL YANG DIDUKUNG UNTUK GENERATE CONTENT ===")
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(f"- {m.name}")

=== DAFTAR MODEL YANG DIDUKUNG UNTUK GENERATE CONTENT ===
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.1-flash-lite-image
- models/gemini-nano-banana-2.1
- models/gemini-3.5-flash
- models/gemini-3.5-flash-lite
- models/gemini-omni-flash-preview
- models/gemini-omni-1.1-flash
- models/gemini-3.5-transcribe
- models/gemini-3.6-flash
- models/gemini-3